# Feature Correlation Analysis

Analyze correlations between engineered features and abuse labels.

In [ ]:
import json
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path

# Load events
data_dir = Path('../backend/data/output')
events = json.load(open(data_dir / 'events.json'))
df = pd.DataFrame(events)

# Aggregate features per account
features = df.groupby('entity_id').agg(
    event_count=('event_type', 'count'),
    refund_count=('event_type', lambda x: (x == 'refund.processed').sum()),
    order_count=('event_type', lambda x: (x == 'order.paid').sum()),
    label=('event_label', lambda x: 1 if 'abuse' in x.values else 0),
    merchant_count=('merchant_id', 'nunique'),
    scenario_count=('scenario_id', 'nunique')
).reset_index()

features['refund_rate'] = features['refund_count'] / features['order_count'].clip(lower=1)
features['events_per_scenario'] = features['event_count'] / features['scenario_count'].clip(lower=1)

print(f'Accounts: {len(features)}')
print(f'Abuse rate: {features["label"].mean():.2%}')
features.head()

## Correlation Matrix

In [ ]:
corr_cols = ['event_count', 'refund_count', 'order_count', 'refund_rate',
             'merchant_count', 'scenario_count', 'events_per_scenario', 'label']

corr = features[corr_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, cmap='coolwarm', center=0, fmt='.2f',
            square=True, linewidths=0.5)
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.savefig('02_feature_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

## Feature Distributions by Label

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

plot_features = ['event_count', 'refund_count', 'order_count', 'refund_rate', 'merchant_count', 'scenario_count']

for ax, feat in zip(axes.flatten(), plot_features):
    for label, color in [(0, 'green'), (1, 'red')]:
        subset = features[features['label'] == label]
        ax.hist(subset[feat], bins=20, alpha=0.5, color=color,
                label='Legitimate' if label == 0 else 'Abuse')
    ax.set_title(feat)
    ax.legend()

plt.suptitle('Feature Distributions by Label', fontsize=14)
plt.tight_layout()
plt.savefig('02_feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()